# Multi RAG Evaluator

## Load Libraries

In [ ]:
import os            # Work with environment variables and file paths
import glob          # Find files using wildcard patterns
import subprocess    # Run external commands or shell processes
import re            # Regular expressions for text pattern matching
from pathlib import Path  # file system paths
import requests      # Send HTTP requests to APIs or websites
import numpy as np   # Numerical computing (arrays, math operations)
from sklearn.manifold import TSNE   # Dimensionality reduction for visualization
import plotly.graph_objects as go   # Interactive plotting
from tqdm import tqdm   # Display progress bars for loops
from pydantic import BaseModel, Field   # Define structured data models with validation
from dotenv import load_dotenv    # Load environment variables from a .env file
from openai import OpenAI        # Official OpenAI client
from litellm import completion   # Lightweight LLM API wrapper
from langchain_ollama import ChatOllama # Ollama lanchain chat
from sentence_transformers import SentenceTransformer #  SentenceTransformer to generate text embeddings
from langchain_huggingface import HuggingFaceEmbeddings  # Create embeddings using HuggingFace models
from chromadb import PersistentClient   # Persistent vector store (Chroma DB)
from langchain_chroma import Chroma # # Chroma vector database integration for LangChain
from IPython.display import Markdown, display  # Markdown libraries 

## Ollma Initialize

In [ ]:
subprocess.Popen("ollama serve", shell=True)
requests.get("http://localhost:11434").content

In [ ]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

## Configuration

In [ ]:
MODEL = "llama3.2"
DB_NAME = "vector_db"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2"

## Represet Documents and Chunks

In [ ]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

In [ ]:
class Chunks(BaseModel):
    chunks: list[Chunk]

## Fetch Documents

In [ ]:
def fetch_documents():    
    # Define a function to fetch documents from the knowledge base

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": folder.name, "source": file.as_posix(), "text": f.read()})
    
    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
documents = fetch_documents()

In [ ]:
documents[0]

## Chunking Prompt

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [ ]:
print(make_prompt(documents[0]))

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
make_messages(documents[0])

## Litellm Process

In [ ]:
def process_document(document):
    messages = make_messages(document)

    response = completion(
        model=f"ollama/{MODEL}",
        messages=messages,
        response_format=Chunks
    )
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

## Langchain Process

In [ ]:
def process_document_lanchain(document):

    ChatOllamaModel = ChatOllama(model=MODEL)
    structured_output = ChatOllamaModel.with_structured_output(Chunks, method='json_schema')
    
    messages = make_messages(document)

    structured_output = structured_output.invoke(messages)
    reply = structured_output.chunks
    return [chunk.as_result(document) for chunk in reply]

In [ ]:
process_document_lanchain(documents[0])

## Create Chunks

In [ ]:
def create_chunks(documents):
    chunks = [chunk for doc in tqdm(documents) for chunk in process_document(doc)]
    return chunks

In [ ]:
documents_mock = documents[:5]

In [ ]:
chunks = create_chunks(documents_mock)
print(len(chunks))

In [ ]:
Markdown(chunks[0].page_content)

## Embeddings

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)

    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)
    
    texts = [chunk.page_content for chunk in chunks]
    emb = SentenceTransformer(model_name_or_path=embedding_model)
    vectors = emb.encode(texts)
    
    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

## Langchain Embeddings

In [ ]:
def create_embeddings_langchain(chunks):
    embedding = HuggingFaceEmbeddings(model_name=embedding_model)

    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embedding).delete_collection()
    
    texts = [chunk.page_content for chunk in chunks]
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    vector_store = Chroma.from_texts(
        texts=texts,
        ids= ids,
        metadatas=metas,
        embedding=embedding,
        persist_directory=DB_NAME
    )

    print(f"Vector store created with {vector_store._collection.count()} documents")

In [ ]:
create_embeddings_langchain(chunks)

## RAG Retrival

In [ ]:
def fetch_context_unranked_langchain(question, k=2):
    embedding = HuggingFaceEmbeddings(model_name=embedding_model)
    vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embedding)
    retriever = vectorstore.as_retriever()
    results = retriever.invoke(question, k=k)

    chunks = []
    for result in results:
        chunks.append(Result(page_content=result.page_content, metadata=result.metadata))
    return chunks

In [ ]:
question = "What are Insurellm primarily operating office?"
chunks = fetch_context_unranked_langchain(question, k=5)

In [ ]:
chunks

## Re-Rank

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    ChatOllamaModel = ChatOllama(model=MODEL)
    structured_output = ChatOllamaModel.with_structured_output(RankOrder, method='json_schema')

    response = structured_output.invoke(messages)
    reply = response.order

    reordered_chunks = [chunks[i] for i in reply]

    return reordered_chunks

In [ ]:
reranked_chunk_ids = rerank(question, chunks)
reranked_chunk_ids

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
def process_document_lanchain(document):

    ChatOllamaModel = ChatOllama(model=MODEL)
    structured_output = ChatOllamaModel.with_structured_output(Chunks, method='json_schema')
    
    messages = make_messages(document)

    structured_output = structured_output.invoke(messages)
    reply = structured_output.chunks
    return [chunk.as_result(document) for chunk in reply]

In [ ]:
def answer_question(question: str, history: str = ""):
    """
    Answer a question using RAG and return the answer and the retrieved context
    """

    un_ranked_chunks = fetch_context_unranked_langchain(question, k=3)
    reranked_chunks = rerank(question, un_ranked_chunks)

    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" 
                          for chunk in reranked_chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    message = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": question}]

    ChatOllamaModel = ChatOllama(model=MODEL)
    response = ChatOllamaModel.invoke(message)

    return response.content

In [ ]:
Markdown(answer_question("What are Insurellm primarily operating office?"))